# HPPCS[04] Multimodal Medical Assistant on Google Colab

This notebook runs the **current** `Codebase/` assistant on Colab (CPU or free **T4 GPU**).

## What the app does
- Upload a **clinical note** (`.txt` / `.md`) + **radiology or pathology** image
- **MedGemma 4B** auto-detects domain and reads the image
- **Llama 3.2** builds triage, findings, references, and follow-up chat
- Evaluation panel shows **label-free** metrics (correlation, robustness, explanation, satisfaction)
- Works on **CPU or GPU** via `--device auto|cpu|gpu` (`runtime_profile.py`)

## Interfaces (same as local)
| Mode | Flag | Use on Colab |
|------|------|----------------|
| **2 — GUI** (browser) | `python main.py --gui` | **Recommended** — open Colab port `8000` |
| 1 — TUI (terminal) | `python main.py --tui` | Awkward in notebooks (needs interactive prompts) |

## Before you start
1. `Runtime` → `Change runtime type` → Hardware accelerator → **T4 GPU** (preferred) or **None** (CPU)
2. Run cells **in order**
3. After the GUI starts, use Colab’s **port 8000** link (or the printed URL)

Compact models (MedGemma 4B + Llama 3.2 3B) keep Colab memory use manageable.

## 1. Check CPU / GPU

`runtime_profile.py` uses `nvidia-smi` when present. GPU is faster for MedGemma; CPU still works (slower).

In [ ]:
import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
    print("GPU available — use --device auto or --device gpu")
else:
    print("No NVIDIA GPU (or nvidia-smi missing). App will run on CPU.")
    print("For faster vision: Runtime → Change runtime type → T4 GPU → Save, then re-run from the top.")

## 2. Install Ollama and start the server

The app talks to Ollama at `http://127.0.0.1:11434` **inside this Colab VM** (not your laptop).

In [ ]:
import os
import subprocess
import time

import requests

if not os.path.exists("/usr/local/bin/ollama"):
    !curl -fsSL https://ollama.com/install.sh | sh

os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"

# Avoid starting a second daemon if one is already up
try:
    if requests.get("http://127.0.0.1:11434/api/tags", timeout=2).status_code == 200:
        print("Ollama already running")
    else:
        raise RuntimeError("not ready")
except Exception:
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=open("/tmp/ollama.log", "ab"),
        stderr=subprocess.STDOUT,
    )
    for _ in range(40):
        try:
            r = requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
            if r.status_code == 200:
                print("Ollama is running")
                break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("Ollama did not start. Check /tmp/ollama.log")

## 3. Get the project code

Expected layout (matches local submission):

```text
capstone/
  Codebase/          # main.py, requirements.txt, …
  sample_data/       # optional demo patient_01…05 files
```

**Option A (recommended for latest local code):** zip the whole `capstone` folder (or at least `Codebase/` + `sample_data/`), upload, unzip.

**Option B:** clone GitHub if that remote already has this rewrite.

In [ ]:
import os
import shutil
import zipfile

from google.colab import files

# --- Option A: upload a zip from your PC (uncomment) ---
# uploaded = files.upload()
# zip_name = next(iter(uploaded))
# zipfile.ZipFile(zip_name).extractall("/content")

# --- Option B: clone GitHub ---
!git clone --depth 1 https://github.com/sumanchatterjeecs2010/capstone_iit.git /content/capstone_iit

PROJECT = None
for candidate in (
    "/content/capstone_iit",
    "/content/capstone",
    "/content",
):
    if os.path.isdir(os.path.join(candidate, "Codebase")):
        PROJECT = candidate
        break

if PROJECT is None and os.path.isdir("/content/Codebase"):
    # Zip contained only Codebase/ — create sibling sample_data if missing
    PROJECT = "/content"
    os.makedirs("/content/sample_data", exist_ok=True)

if PROJECT is None:
    raise FileNotFoundError(
        "Could not find Codebase/. Upload a zip of the capstone folder or fix the clone URL."
    )

CODEBASE = os.path.join(PROJECT, "Codebase")
os.chdir(CODEBASE)
print("Project root:", PROJECT)
print("Working directory:", os.getcwd())
print("sample_data exists:", os.path.isdir(os.path.join(PROJECT, "sample_data")))
print("Codebase files:", sorted(os.listdir("."))[:20])

## 4. Install Python packages and pull models

- `medgemma:4b` ≈ 3–4 GB (vision)
- `llama3.2:3b` ≈ 2 GB (text)

The app **unloads MedGemma after vision** so Llama can use VRAM/RAM (`runtime_profile.py`).
Low-RAM fallback: `ollama pull llama3.2:1b` and run with `--model llama3.2:1b`.

In [ ]:
!pip -q install -r requirements.txt
!ollama pull medgemma:4b
!ollama pull llama3.2:3b
!ollama list

# Sanity-check runtime detection (same logic as main.py)
from runtime_profile import describe_startup, get_profile

print(describe_startup(get_profile("auto")))

## 5. Run the GUI (recommended on Colab)

Starts the FastAPI dashboard (`upload_app.py`) — same as local `python main.py --gui`.

1. Run the cell below (it blocks while the server runs)
2. In Colab: open the **port 8000** URL from the output / Ports panel
3. Upload a **radiology or pathology** image + note (domain is auto-detected)
4. Review triage, findings, references, Evaluation metrics, and chat

Use `--device auto` (default). Force GPU/CPU if needed:
- `python main.py --gui --device gpu`
- `python main.py --gui --device cpu`

In [ ]:
# Expose port 8000 in Colab (ignore errors on older Colab builds)
try:
    from google.colab import output

    output.serve_kernel_port_as_window(8000)
    print("If a window opens, use that URL for the GUI.")
except Exception as exc:
    print("Port helper:", exc)
    print("Use Colab Ports / the URL printed by uvicorn for port 8000.")

# Mode 2 GUI — auto CPU/GPU via runtime_profile
!python main.py --gui --device auto --port 8000 --host http://127.0.0.1:11434

## 6. Optional: one-off CLI case (no browser)

Uses files under `../sample_data/` (sibling of `Codebase/`). Skip if you only use the GUI.

Do **not** run this while the GUI cell above is still running — interrupt that cell first.

In [ ]:
import os

note = "../sample_data/patient_01.txt"
image = "../sample_data/patient_01.jpg"

if os.path.exists(note) and os.path.exists(image):
    !python main.py --device auto --text ../sample_data/patient_01.txt --image ../sample_data/patient_01.jpg
else:
    print("Demo files missing. Upload via GUI, or add sample_data/ next to Codebase/.")
    print("Expected:", os.path.abspath(note), os.path.abspath(image))

## 7. Download session outputs to your PC

GUI sessions write under `Codebase/uploads/processed/<session_id>/conversation.json`.
CLI one-off writes `uploads/processed/conversation_upload.json`.

In [ ]:
import glob

from google.colab import files

paths = sorted(
    glob.glob("uploads/processed/**/conversation.json", recursive=True)
    + glob.glob("uploads/processed/conversation_upload.json")
)

if not paths:
    print("No conversation JSON found yet. Run the GUI or CLI cell first.")
else:
    for path in paths:
        print("Downloading", path)
        files.download(path)

## Troubleshooting

| Issue | Fix |
|-------|-----|
| Ollama not reachable | Re-run section 2; check `/tmp/ollama.log` |
| Model missing | Re-run `ollama pull medgemma:4b` and `ollama pull llama3.2:3b` |
| Out-of-scope image | Use radiology (X-ray/CT/MRI) or histopathology slide only |
| Slow on CPU | Switch runtime to T4 GPU and re-run from section 1 |
| `--device gpu` falls back to CPU | No GPU attached — change runtime type |
| Cannot find Codebase | Upload zip of full `capstone/` (with `Codebase/` + optional `sample_data/`) |
| Port 8000 not opening | Open Colab **Ports** panel or use the URL printed by the server |